# SQL Interview Exercises for Quantitative Research

15 progressive exercises.

**Format:** inspect the provided tables, read the requested result, then write the SQL inside `con.sql(...).df()`.


## Setup

The data model is the same as in the SQL mastery notebook:

- `assets`: asset metadata;
- `prices`: daily close prices and volumes;
- `signals`: daily model signals;
- `trades`: example executions.

Run the setup once, then work only in the TODO cells.

In [2]:
# Run once if DuckDB is not installed:
# %pip install duckdb

import duckdb
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
con = duckdb.connect()

dates = pd.bdate_range("2024-01-02", periods=90)

assets = pd.DataFrame({
    "asset": ["EQ_US", "EQ_EU", "BOND_US", "BOND_EU", "GOLD", "OIL", "EURUSD", "USDJPY"],
    "asset_class": ["Equity", "Equity", "Rates", "Rates", "Commodity", "Commodity", "FX", "FX"],
    "region": ["US", "Europe", "US", "Europe", "Global", "Global", "Europe", "Japan"],
    "currency": ["USD", "EUR", "USD", "EUR", "USD", "USD", "USD", "JPY"],
})

rows = []
signal_rows = []

for j, asset in enumerate(assets["asset"]):
    rets = rng.normal(
        0.0002 + j * 0.00001,
        0.008 + j * 0.0003,
        len(dates),
    )
    close = (100 + 5 * j) * np.exp(np.cumsum(rets))
    volume = rng.integers(100_000, 2_000_000, len(dates))
    signal = (
        pd.Series(rets)
        .rolling(10, min_periods=3)
        .mean()
        .to_numpy()
        + rng.normal(0, 0.002, len(dates))
    )

    rows.extend(zip(dates, [asset] * len(dates), close, volume))
    signal_rows.extend(zip(dates, [asset] * len(dates), signal))

prices = pd.DataFrame(
    rows,
    columns=["date", "asset", "close", "volume"],
)

signals = pd.DataFrame(
    signal_rows,
    columns=["date", "asset", "signal"],
)

trade_dates = dates[::9]

trades = pd.DataFrame({
    "trade_id": np.arange(1, len(trade_dates) * 3 + 1),
    "date": np.repeat(trade_dates, 3),
    "asset": np.tile(["EQ_US", "GOLD", "EURUSD"], len(trade_dates)),
    "quantity": rng.integers(-250, 251, len(trade_dates) * 3),
    "price": rng.normal(100, 8, len(trade_dates) * 3),
})

for df in [assets, prices, signals, trades]:
    string_cols = df.select_dtypes(include=["str", "string"]).columns
    df[string_cols] = df[string_cols].astype("object")

con.register("assets_df", assets)
con.register("prices_df", prices)
con.register("signals_df", signals)
con.register("trades_df", trades)

con.execute("CREATE OR REPLACE TABLE assets AS SELECT * FROM assets_df")
con.execute("CREATE OR REPLACE TABLE prices AS SELECT * FROM prices_df")
con.execute("CREATE OR REPLACE TABLE signals AS SELECT * FROM signals_df")
con.execute("CREATE OR REPLACE TABLE trades AS SELECT * FROM trades_df")

## Available tables

In [3]:
display(assets)
display(prices.head())
display(signals.head())
display(trades.head())

,asset,asset_class,region,currency
0,EQ_US,Equity,US,USD
1,EQ_EU,Equity,Europe,EUR
2,BOND_US,Rates,US,USD
3,BOND_EU,Rates,Europe,EUR
4,GOLD,Commodity,Global,USD
5,OIL,Commodity,Global,USD
6,EURUSD,FX,Europe,USD
7,USDJPY,FX,Japan,JPY


,date,asset,close,volume
0,2024-01-02,EQ_US,100.264122,1041689
1,2024-01-03,EQ_US,99.453286,947696
2,2024-01-04,EQ_US,100.072173,415806
3,2024-01-05,EQ_US,100.848176,823940
4,2024-01-08,EQ_US,99.306189,553655


,date,asset,signal
0,2024-01-02,EQ_US,NaN
1,2024-01-03,EQ_US,NaN
2,2024-01-04,EQ_US,0.001917
3,2024-01-05,EQ_US,0.006105
4,2024-01-08,EQ_US,0.004435


,trade_id,date,asset,quantity,price
0,1,2024-01-02,EQ_US,-178,84.632673
1,2,2024-01-02,GOLD,-203,103.816281
2,3,2024-01-02,EURUSD,44,99.995495
3,4,2024-01-15,EQ_US,-250,100.317548
4,5,2024-01-15,GOLD,73,107.685874


## 1. Filter, expression, sort, limit — Foundation

From `prices`, return the **10 observations with the highest dollar volume** during February 2024 for `EQ_US` and `EQ_EU`.

Create:

`date, asset, close, volume, dollar_volume`

where `dollar_volume = close * volume`.

Order by `dollar_volume` descending.

In [4]:
# TODO
result_01 = con.sql('''
SELECT 
    date,
    asset,
    close,
    volume,
    close * volume AS dollar_volume
FROM prices
WHERE asset in ('EQ_US', 'EQ_EU')
ORDER BY dollar_volume DESC
LIMIT 10
''').df()

result_01

,date,asset,close,volume,dollar_volume
0,2024-04-23,EQ_US,103.698323,1980399,2.053641e+08
1,2024-02-21,EQ_EU,102.428347,1993653,2.042066e+08
2,2024-04-04,EQ_EU,107.179086,1869348,2.003550e+08
3,2024-02-09,EQ_EU,103.470978,1923883,1.990661e+08
4,2024-02-12,EQ_EU,103.282233,1921872,1.984952e+08
5,2024-05-06,EQ_US,102.526799,1921262,1.969808e+08
6,2024-01-24,EQ_EU,102.522554,1909771,1.957946e+08
7,2024-03-26,EQ_EU,105.879594,1848223,1.956891e+08
8,2024-02-26,EQ_EU,102.619202,1897451,1.947149e+08
9,2024-02-02,EQ_US,100.114928,1941435,1.943666e+08


## 2. DISTINCT and filtering — Foundation

From `assets`, return the distinct currencies used by assets whose `asset_class` is either `Equity` or `Rates`.

Return one column:

`currency`

sorted alphabetically.

In [5]:
# TODO
result_02 = con.sql('''
SELECT DISTINCT(currency) AS currency
FROM assets
WHERE asset_class IN ('Equity', 'Rates')
ORDER BY currency
''').df()

result_02

,currency
0,EUR
1,USD


## 3. NULL and COALESCE — Foundation

From `signals`, find rows where `signal` is missing.

Return:

`date, asset, signal, signal_filled`

where `signal_filled` replaces missing signals with `0.0`.

Order by `date, asset` and keep the first 10 rows.

In [6]:
# TODO
result_03 = con.sql('''
SELECT 
    date,
    asset,
    signal,
    COALESCE(signal, 0.0) AS signal_filled
FROM signals
WHERE signal IS NULL
ORDER BY date, asset
LIMIT 10
''').df()

result_03

,date,asset,signal,signal_filled
0,2024-01-02,BOND_EU,NaN,0.0
1,2024-01-02,BOND_US,NaN,0.0
2,2024-01-02,EQ_EU,NaN,0.0
3,2024-01-02,EQ_US,NaN,0.0
4,2024-01-02,EURUSD,NaN,0.0
5,2024-01-02,GOLD,NaN,0.0
6,2024-01-02,OIL,NaN,0.0
7,2024-01-02,USDJPY,NaN,0.0
8,2024-01-03,BOND_EU,NaN,0.0
9,2024-01-03,BOND_US,NaN,0.0


## 4. CASE — Foundation

From non-null rows of `signals`, classify each signal as:

- `Positive` if `signal > 0.002`;
- `Negative` if `signal < -0.002`;
- `Neutral` otherwise.

Return:

`date, asset, signal, signal_bucket`

ordered by `date, asset`, limited to 15 rows.

In [7]:
# TODO
result_04 = con.sql('''
SELECT
    date,
    asset,
    signal,
    CASE
        WHEN signal > 0.002 THEN 'Positive'
        WHEN signal < -0.002 THEN 'Negative'
        ELSE 'Neutral'
    END AS signal_bucket
FROM signals
WHERE signal IS NOT NULL
ORDER BY date, asset
LIMIT 15
''').df()

result_04

,date,asset,signal,signal_bucket
0,2024-01-04,BOND_EU,-0.005625,Negative
1,2024-01-04,BOND_US,0.001994,Neutral
2,2024-01-04,EQ_EU,0.012694,Positive
3,2024-01-04,EQ_US,0.001917,Neutral
4,2024-01-04,EURUSD,0.000013,Neutral
5,2024-01-04,GOLD,-0.002603,Negative
6,2024-01-04,OIL,-0.002896,Negative
7,2024-01-04,USDJPY,-0.001788,Neutral
8,2024-01-05,BOND_EU,-0.001388,Neutral
9,2024-01-05,BOND_US,0.005337,Positive


## 5. GROUP BY — Foundation

For each asset in `prices`, calculate:

`asset, avg_close, close_std, avg_volume, n_obs`

using the same aggregate definitions as the mastery notebook.

Order by `avg_volume` descending.

In [8]:
# TODO
result_05 = con.sql('''
SELECT
    asset,
    AVG(close) AS avg_close,
    STDDEV_SAMP(close) AS close_std,
    AVG(volume) AS avg_volume,
    COUNT(*) AS n_obs
FROM prices
GROUP BY asset
ORDER BY avg_volume DESC
''').df()

result_05

,asset,avg_close,close_std,avg_volume,n_obs
0,GOLD,123.959129,3.734945,1.132150e+06,90
1,EQ_EU,104.912675,2.200180,1.127521e+06,90
2,BOND_US,115.476783,2.534195,1.117218e+06,90
3,EURUSD,136.086343,5.262179,1.102091e+06,90
4,USDJPY,128.854377,3.528672,1.099388e+06,90
5,OIL,119.892677,3.802344,1.098402e+06,90
6,BOND_EU,112.660750,4.603724,1.013987e+06,90
7,EQ_US,101.965246,1.974401,1.001868e+06,90


## 6. WHERE versus HAVING — Intermediate

Using only observations on or after `2024-02-01`, calculate average volume and observation count for each asset.

Keep only assets whose average volume is greater than `1,000,000`.

Return:

`asset, avg_volume, n_obs`

ordered by `avg_volume` descending.

In [9]:
# TODO
result_06 = con.sql('''
SELECT 
    asset,
    AVG(volume) AS avg_volume,
    COUNT(*) AS n_obs
FROM prices
WHERE date >= '2024-02-01'
GROUP BY asset
HAVING avg_volume > 1_000_000
ORDER BY avg_volume DESC
''').df()

result_06

,asset,avg_volume,n_obs
0,OIL,1.144554e+06,68
1,EQ_EU,1.142882e+06,68
2,BOND_US,1.114117e+06,68
3,GOLD,1.105207e+06,68
4,EURUSD,1.103920e+06,68
5,USDJPY,1.094048e+06,68
6,EQ_US,1.005465e+06,68


## 7. JOIN — Intermediate

Join `prices` with `assets`.

Keep only `Commodity` assets and return:

`date, asset, region, close, volume`

ordered by `date, asset`, limited to 20 rows.

In [10]:
# TODO
result_07 = con.sql('''
SELECT 
    p.date AS date,
    p.asset AS asset,
    a.region AS region,
    p.close AS close,
    p.volume AS volume
FROM prices AS p
INNER JOIN assets as a
    ON p.asset = a.asset
    WHERE a.asset_class = 'Commodity'
ORDER BY date, asset
LIMIT 20
''').df()

result_07

,date,asset,region,close,volume
0,2024-01-02,GOLD,Global,119.284618,1977145
1,2024-01-02,OIL,Global,123.862463,750289
2,2024-01-03,GOLD,Global,118.370443,954302
3,2024-01-03,OIL,Global,124.424586,603133
4,2024-01-04,GOLD,Global,118.921631,650360
5,2024-01-04,OIL,Global,124.459371,1865745
6,2024-01-05,GOLD,Global,117.281474,1795726
7,2024-01-05,OIL,Global,123.606867,1259921
8,2024-01-08,GOLD,Global,117.730954,1518165
9,2024-01-08,OIL,Global,123.361087,895514


## 8. JOIN + correlated subquery — Intermediate

For each non-null signal observation, keep it only if its `signal` is **above that asset's own full-sample average signal**.

Also attach the asset class.

Use `signals` as the main table, join `assets`, and use a correlated subquery for the asset-specific average.

Return:

`date, asset, asset_class, signal`

ordered by `asset, date`, limited to 20 rows.

In [11]:
# TODO
result_08 = con.sql('''
SELECT
    s.date AS date,
    s.asset AS asset,
    a.asset_class AS asset_class,
    s.signal AS signal
FROM signals AS s
LEFT JOIN assets AS a
    ON s.asset = a.asset
WHERE signal IS NOT NULL 
AND signal > (
    SELECT AVG(s2.signal)
    FROM signals AS s2
    WHERE s2.asset = s.asset
) 
ORDER BY asset, date
LIMIT 20
''').df()

result_08

,date,asset,asset_class,signal
0,2024-01-09,BOND_EU,Rates,0.001758
1,2024-01-10,BOND_EU,Rates,0.003571
2,2024-01-11,BOND_EU,Rates,0.002134
3,2024-01-12,BOND_EU,Rates,0.001397
4,2024-01-15,BOND_EU,Rates,0.003761
5,2024-01-16,BOND_EU,Rates,0.002648
6,2024-01-17,BOND_EU,Rates,0.005689
7,2024-01-18,BOND_EU,Rates,0.006455
8,2024-01-19,BOND_EU,Rates,0.002499
9,2024-01-22,BOND_EU,Rates,0.003078


## 9. CTE + dates + JOIN — Intermediate

Create a monthly summary by asset class.

Join `prices` with `assets`, use `DATE_TRUNC('month', date)`, and calculate:

`month_start, asset_class, avg_close, avg_volume`

for each month and asset class.

Use a `WITH` CTE for the joined data.

Order by `month_start, asset_class`.

In [15]:
# TODO
result_09 = con.sql('''
WITH full_table AS (
    SELECT 
        p.date AS date,
        a.asset_class AS asset_class,
        p.close AS close,
        p.volume AS volume
    FROM prices as p
    INNER JOIN assets as a
    ON p.asset = a.asset
    ORDER BY date, asset_class
)
SELECT
    DATE_TRUNC('month', date) AS month_start,
    asset_class,
    AVG(close) AS avg_close,
    AVG(volume) AS avg_volume
FROM full_table 
GROUP BY month_start, asset_class
ORDER BY month_start, asset_class
''').df()

result_09

,month_start,asset_class,avg_close,avg_volume
0,2024-01-01,Commodity,121.213208,1.085589e+06
1,2024-01-01,Equity,102.018365,1.035396e+06
2,2024-01-01,FX,131.947181,1.106166e+06
3,2024-01-01,Rates,116.266732,1.094839e+06
4,2024-02-01,Commodity,123.322434,1.086991e+06
5,2024-02-01,Equity,102.358334,1.171672e+06
6,2024-02-01,FX,129.497637,1.105033e+06
7,2024-02-01,Rates,115.161421,1.043145e+06
8,2024-03-01,Commodity,122.249665,1.099402e+06
9,2024-03-01,Equity,104.836744,9.979753e+05


## 10. LAG — Window functions

For every asset, calculate the previous close and daily simple return:

`simple_return = close / previous_close - 1`

Return:

`date, asset, close, previous_close, simple_return`

using `LAG` partitioned by asset and ordered by date.

Order by `asset, date`, limited to 20 rows.

In [17]:
# TODO
result_10 = con.sql('''
WITH prev_close AS (
    SELECT
        date,
        asset,
        close,
        LAG(close) OVER(
            PARTITION BY asset
            ORDER BY date
        ) AS previous_close
    FROM prices
)
SELECT
    date, 
    asset,
    close,
    previous_close,
    close/previous_close - 1 AS simple_return
FROM prev_close
ORDER BY asset, date
LIMIT 20
''').df()

result_10

,date,asset,close,previous_close,simple_return
0,2024-01-02,BOND_EU,115.305626,NaN,NaN
1,2024-01-03,BOND_EU,114.601711,115.305626,-0.006105
2,2024-01-04,BOND_EU,113.292497,114.601711,-0.011424
3,2024-01-05,BOND_EU,114.164642,113.292497,0.007698
4,2024-01-08,BOND_EU,114.546500,114.164642,0.003345
5,2024-01-09,BOND_EU,117.028327,114.546500,0.021667
6,2024-01-10,BOND_EU,117.493815,117.028327,0.003978
7,2024-01-11,BOND_EU,117.927055,117.493815,0.003687
8,2024-01-12,BOND_EU,117.779071,117.927055,-0.001255
9,2024-01-15,BOND_EU,118.665653,117.779071,0.007527


## 11. RANK and NTILE — Window functions

Using non-null rows from `signals`, rank signals cross-sectionally for each date.

Return:

`date, asset, signal, signal_rank, signal_quartile`

where:

- `signal_rank` uses `RANK()`;
- `signal_quartile` uses `NTILE(4)`.

Both windows should partition by date and order by signal ascending.

Order by `date, signal_rank`, limited to 20 rows.

In [ ]:
# TODO
result_11 = con.sql('''

''').df()

result_11

## 12. Rolling window — Window functions

Calculate daily simple returns first, then compute the **20-observation rolling sample volatility** of returns for each asset.

Return:

`date, asset, ret, rolling_vol_20`

Use a CTE for returns and:

`ROWS BETWEEN 19 PRECEDING AND CURRENT ROW`

for the rolling window.

Order by `asset, date`, limited to 30 rows.

In [ ]:
# TODO
result_12 = con.sql('''

''').df()

result_12

## 13. JOIN + RANK + QUALIFY — Advanced

For each `date` and `asset_class`, keep the **two highest non-null signals**.

Join `signals` with `assets`, then calculate a descending rank within each `(date, asset_class)` group.

Return:

`date, asset_class, asset, signal, rank_desc`

and use `QUALIFY` to keep ranks `<= 2`.

Order by `date, asset_class, rank_desc`, limited to 30 rows.

In [ ]:
# TODO
result_13 = con.sql('''

''').df()

result_13

## 14. Subquery + multiple JOINs + aggregation — Final

For each asset class, count how many price observations have volume **above that asset's own full-sample average volume**.

Build a subquery that calculates:

`asset, avg_volume`

from `prices`, then join it back to `prices` and join `assets`.

Return:

`asset_class, n_above_avg_volume, avg_excess_volume`

where:

`avg_excess_volume = AVG(volume - avg_volume)`

using only rows where `volume > avg_volume`.

Order by `n_above_avg_volume` descending.

In [ ]:
# TODO
result_14 = con.sql('''

''').df()

result_14

## 15. Multi-stage research query — Final proficiency check

For each asset class, identify the asset with the **highest full-sample average daily simple return**.

Build the query in stages:

1. calculate daily simple returns from `prices` with `LAG`;
2. calculate each asset's `mean_ret` and sample volatility `vol`;
3. join the asset statistics to `assets`;
4. rank assets within each `asset_class` by `mean_ret` descending;
5. keep only rank `1`.

Return:

`asset_class, asset, mean_ret, vol, rank_in_class`

ordered by `asset_class`.

Use CTEs and/or subqueries as needed.

In [ ]:
# TODO
result_15 = con.sql('''

''').df()

result_15